In [1]:
# =============================================================================
# Q-LEARNING, SARSA, DQN — learn WHILE you walk (not only at the finish line)
# =============================================================================
#
# Last notebook (RL foundations):
#   Dynamic Programming → needs the full rulebook P (model-based planning)
#   Monte Carlo         → learns from COMPLETE episodes (wait until the end)
#
# This notebook: Temporal Difference (TD) methods.
#   Update your guess AFTER EACH STEP, using the reward you just got PLUS
#   your current guess of the next state's value. No rulebook. No waiting
#   for the episode to finish.
#
# Analogy — grading a hike as you go:
#   Monte Carlo: finish the trail, then score every mile from total time.
#   TD:         at each milepost, update "how good is this spot?" using
#               (time so far) + (my estimate of the remaining trail).
#
# That "estimate of the remaining trail" is bootstrapping — learning from
# your own current estimates. Powerful, a bit like standing on your own
# shoulders. Works online, step by step.
#


# -----------------------------------------------------------------------------
# 1. TEMPORAL DIFFERENCE (TD) LEARNING — the big idea
# -----------------------------------------------------------------------------
#
# Recall: return from time t is the discounted sum of future rewards:
#   G_t = R_{t+1} + γ R_{t+2} + γ² R_{t+3} + …
#
# Monte Carlo waits for the whole G_t, then averages.
#
# TD(0) for state values instead uses a ONE-STEP target:
#   target ≈ R_{t+1} + γ V(S_{t+1})     ← "reward now + guess of what comes next"
#
# Update:
#   V(S_t) ← V(S_t) + α [ target − V(S_t) ]
#
# The thing in brackets is the TD error δ:
#   δ = (what just happened + my guess for next) − (what I thought before)
#
# If δ > 0: "that step was BETTER than I expected → raise V(S_t)"
# If δ < 0: "worse than expected → lower V(S_t)"
#
# α (learning rate): how big a step toward the new target (e.g. 0.1).
# γ (discount): how much future rewards matter (same as before).
#
# For CONTROL (picking actions) we track Q(s,a) = "how good is action a in s?"
# Two famous recipes: SARSA and Q-learning. They differ in ONE word of the
# target — and that one word changes their personality.
#


# -----------------------------------------------------------------------------
# 2. SARSA — On-policy TD control  (State → Action → Reward → State → Action)
# -----------------------------------------------------------------------------
#
# Name comes from the tuple it uses each update:
#   (S, A, R, S', A')
#
# After taking A in S, seeing R and landing in S', SARSA ALSO picks the NEXT
# action A' the SAME way it usually acts (e.g. ε-greedy), then updates:
#
#   Q(S,A) ← Q(S,A) + α [ R + γ Q(S', A') − Q(S,A) ]
#                              └─────┬─────┘
#                         value of the action we WILL take next
#
# On-policy = "learn about the policy you are actually following."
# Including the exploration! If ε-greedy sometimes walks off a cliff,
# SARSA learns "near the cliff, that Q is dangerous" because A' might be
# the clumsy exploratory step.
#
# Personality: cautious. Good when exploration is risky in the real world
# (robots, medicine). The learned policy matches "how I behave while learning."
#


# -----------------------------------------------------------------------------
# 3. Q-LEARNING — Off-policy TD control  (dream of the best next move)
# -----------------------------------------------------------------------------
#
# Same setup: take A in S (often still ε-greedy for exploration), see R, S'.
# But the TARGET pretends the NEXT action is the BEST one, not the one you
# might actually take:
#
#   Q(S,A) ← Q(S,A) + α [ R + γ max_{a'} Q(S', a') − Q(S,A) ]
#                              └──────────┬──────────┘
#                         value of the BEST action available next
#
# Off-policy = "learn about a DIFFERENT (greedy) policy while behaving with
# exploration." Behavior policy explores; target policy is greedy.
#
# Personality: optimistic / bold. Learns the optimal path even if while
# training you sometimes take dumb exploratory moves. Classic cliff-walk
# demo: Q-learning hugs the cliff edge (shortest path); SARSA stays safer
# inland (because it "fears" its own ε-slips).
#
# Tiny comparison table:
#
#   Method       Target uses              Learns about          Typical vibe
#   -----------  -----------------------  --------------------  -------------
#   SARSA        Q(S', A') you will take  the exploring policy  cautious
#   Q-learning   max_a Q(S', a')          the greedy optimal    ambitious
#


# -----------------------------------------------------------------------------
# 4. DEEP Q-NETWORK (DQN) — Q-learning when the table does not fit
# -----------------------------------------------------------------------------
#
# Tabular Q: one number per (state, action). Fine for a 4×4 grid.
# Broken for Atari pixels or big continuous spaces — too many states.
#
# DQN idea: replace the table with a neural net Q(s, a; θ) that OUTPUTS
# action-values from a state (e.g. image → 4 joystick scores).
#
# Still the Q-learning target, but now a regression loss:
#   y = R + γ max_{a'} Q(S', a'; θ⁻)     (θ⁻ = a frozen "target network")
#   loss ≈ (y − Q(S, A; θ))²
#
# Two tricks that made deep RL actually work (Mnih et al., 2015):
#
#   (1) Experience replay
#       Store past transitions (S,A,R,S') in a big buffer. Train on RANDOM
#       mini-batches from the buffer — breaks the "correlated consecutive
#       frames" problem, like shuffling a dataset.
#
#   (2) Target network
#       Keep a slow-copy θ⁻ of the net for computing y. Update θ⁻ only
#       every N steps (or soft-update). Stops the moving-target chase
#       where the thing you chase is also the thing you train.
#
# Rough mental model:
#   Tabular Q-learning = flashcards for every room-door pair.
#   DQN               = a brain that looks at the room and guesses door scores.
#


# -----------------------------------------------------------------------------
# HOW THE PIECES FIT (roadmap for this notebook)
# -----------------------------------------------------------------------------
#
#   MC (previous)     wait for full return G          stable, slow, needs episodes
#   TD / SARSA        bootstrap with Q(S', A')        online, on-policy
#   Q-learning        bootstrap with max Q(S', ·)     online, off-policy
#   DQN               Q-learning + neural net         scales to big / pixel states
#
# Same goal as always: find a good policy π that maximizes discounted return.
# Different tools for when you get the learning signal (end vs each step) and
# how you represent Q (table vs network).
#
# Next cells: implement SARSA & Q-learning on GridWorld, compare them, then
# a small DQN sketch so the "table → network" jump feels concrete.
#


In [ ]:
# =============================================================================
# SETUP + GRIDWORLD — the tiny world SARSA / Q-learning / DQN will play in
# =============================================================================
#
# Same maze idea as Day 22 (copied here so this notebook runs alone):
#
#     (0,0) (0,1) (0,2) (0,3)
#     (1,0)  ##   (1,2) (1,3)      ## = wall
#     (2,0) (2,1)  ##   (2,3)
#     (3,0) (3,1) (3,2) [GOAL]
#
# Every step costs -1. Goal ends the episode. Short paths = higher return.
#
# For TD control we mostly need reset() / step() — play one move at a time.
# We keep P around for optional DP checks; SARSA/Q-learning will NOT read it.
#
# Error fixed below: `import gym` failed (package not installed, and we do not
# need it for GridWorld). Torch is optional here too — only required later
# for DQN. Tabular SARSA / Q-learning need only numpy.
#

import numpy as np
import random
import matplotlib.pyplot as plt
from collections import defaultdict

# Seeds so re-runs look similar (still some randomness in exploration later)
np.random.seed(42)
random.seed(42)

# Torch is for the DQN cell later — import softly so tabular cells still run
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    import torch.nn.functional as F

    torch.manual_seed(42)
    TORCH_OK = True
    print(f"PyTorch {torch.__version__} ready (for DQN later).")
except ImportError:
    TORCH_OK = False
    print("PyTorch not installed — tabular SARSA/Q-learning still work.")

# Classic `gym` is optional / often missing. Prefer gymnasium if present.
# GridWorld does not use either; only a future Atari-style DQN demo would.
try:
    import gymnasium as gym  # modern fork

    GYM_OK = True
    print("gymnasium available (optional).")
except ImportError:
    try:
        import gym  # older package name

        GYM_OK = True
        print("gym available (optional).")
    except ImportError:
        GYM_OK = False
        print("No gym/gymnasium — fine for this notebook's GridWorld demos.")


class GridWorld:
    """
    Tiny grid MDP with a Gym-style API:

      state  = (row, col)
      action = 'up' | 'down' | 'left' | 'right'
      step   → (next_state, reward, done, info)

    Bounce off walls/obstacles (stay put, still pay -1).
    """

    def __init__(self, size=4, obstacles=None, terminal=None, gamma=0.9):
        self.size = size
        # Every cell is a potential state (hashable tuple → good dict key for Q)
        self.states = [(i, j) for i in range(size) for j in range(size)]
        self.terminal = terminal if terminal is not None else (size - 1, size - 1)
        self.obstacles = list(obstacles) if obstacles is not None else [(1, 1), (2, 2)]

        self.actions = ["up", "down", "left", "right"]
        # How each action nudges (row, col). "up" decreases row (toward top).
        self.action_effects = {
            "up": (-1, 0),
            "down": (1, 0),
            "left": (0, -1),
            "right": (0, 1),
        }

        self.gamma = gamma
        self.reward_step = -1.0  # living cost → prefer fewer steps to goal
        self.reward_terminal = 0.0

        # Optional model P[s][a] = [(prob, next_s, reward, done), ...]
        # Deterministic here (prob=1). TD methods ignore this and use step().
        self.P = {}
        for s in self.states:
            if s == self.terminal or s in self.obstacles:
                continue
            self.P[s] = {}
            for a in self.actions:
                next_s = self._get_next_state(s, a)
                done = next_s == self.terminal
                self.P[s][a] = [(1.0, next_s, self.reward_step, done)]

    def _get_next_state(self, state, action):
        """Physics of one move. Illegal → stay in place."""
        if state == self.terminal or state in self.obstacles:
            return state
        i, j = state
        di, dj = self.action_effects[action]
        ni, nj = i + di, j + dj
        if 0 <= ni < self.size and 0 <= nj < self.size and (ni, nj) not in self.obstacles:
            return (ni, nj)
        return state  # hit wall / obstacle → bounce

    def reset(self):
        """Start a new episode from a random free cell (for learning)."""
        available = [
            s for s in self.states if s != self.terminal and s not in self.obstacles
        ]
        self.current_state = random.choice(available)
        return self.current_state

    def step(self, action):
        """
        Take one action. Returns Gym-style tuple:
          next_state, reward, done, info

        SARSA / Q-learning loop:
          s = env.reset()
          while not done:
              a = pick_action(s)
              s, r, done, _ = env.step(a)
              # TD update using (s, a, r, s')
        """
        next_s = self._get_next_state(self.current_state, action)
        reward = self.reward_step
        done = next_s == self.terminal
        self.current_state = next_s
        return next_s, reward, done, {}


# --- sanity check: env works without gym ------------------------------------
env = GridWorld(size=4)
print(f"\nGridWorld size={env.size}  states={len(env.states)}  free={len(env.P)}")
print(f"Terminal={env.terminal}  Obstacles={env.obstacles}")
print("Example P[(0,0)]['right']:", env.P[(0, 0)]["right"])

s = env.reset()
print(f"Episode demo — start {s}")
for _ in range(4):
    a = random.choice(env.actions)
    s, r, done, _ = env.step(a)
    print(f"  {a:5s} → {s}  reward={r}  done={done}")
    if done:
        break

# Picture of the board
board = np.zeros((env.size, env.size))
for oi, oj in env.obstacles:
    board[oi, oj] = -1
board[env.terminal] = 2
plt.figure(figsize=(4, 4))
plt.imshow(board, cmap="RdYlGn", vmin=-1, vmax=2)
for i in range(env.size):
    for j in range(env.size):
        if (i, j) in env.obstacles:
            plt.text(j, i, "X", ha="center", va="center", color="white", fontsize=14)
        elif (i, j) == env.terminal:
            plt.text(j, i, "G", ha="center", va="center", fontweight="bold", fontsize=14)
        else:
            plt.text(j, i, f"{i},{j}", ha="center", va="center", color="gray", fontsize=8)
plt.title("GridWorld — green=goal, red=wall")
plt.axis("off")
plt.show()

# ---------------------------------------------------------------------------
# HOW TO READ
# ---------------------------------------------------------------------------
# reset/step  → what SARSA & Q-learning call every episode (model-free play).
# P           → optional rulebook; not required for TD (unlike DP).
# reward -1   → optimal policy reaches G in as few steps as possible.
# TORCH_OK / GYM_OK → flags for later cells; missing gym no longer crashes setup.
#


In [3]:
# Tabular Q‑Learning

def q_learning(env, num_episodes=500, alpha=0.1, gamma=0.9, epsilon=0.1, epsilon_decay=0.995, min_epsilon=0.01):
    Q = defaultdict(lambda: np.zeros(len(env.actions)))
    # For convergence tracking
    episode_rewards = []
    for ep in range(num_episodes):
        state = env.reset()
        done = False
        total_reward = 0
        while not done:
            # Epsilon-greedy
            if random.random() < epsilon:
                action_idx = random.randrange(len(env.actions))
            else:
                action_idx = np.argmax(Q[state])
            action = env.actions[action_idx]
            next_state, reward, done, _ = env.step(action)
            # Q-learning update
            best_next = np.max(Q[next_state]) if next_state != env.terminal else 0.0
            td_target = reward + gamma * best_next
            td_error = td_target - Q[state][action_idx]
            Q[state][action_idx] += alpha * td_error
            state = next_state
            total_reward += reward
        episode_rewards.append(total_reward)
        epsilon = max(min_epsilon, epsilon * epsilon_decay)
    # Derive deterministic policy
    policy = {}
    for s in env.states:
        if s != env.terminal and s not in env.obstacles:
            policy[s] = env.actions[np.argmax(Q[s])]
    return Q, policy, episode_rewards

In [ ]:
# Tabular SARSA

def sarsa(env, num_episodes=500, alpha=0.1, gamma=0.9, epsilon=0.1, epsilon_decay=0.995, min_epsilon=0.01):
    Q = defaultdict(lambda: np.zeros(len(env.actions)))
    episode_rewards = []
    for ep in range(num_episodes):
        state = env.reset()
        done = False
        # Choose initial action (epsilon-greedy)
        if random.random() < epsilon:
            action_idx = random.randrange(len(env.actions))
        else:
            action_idx = np.argmax(Q[state])
        total_reward = 0
        while not done:
            action = env.actions[action_idx]
            next_state, reward, done, _ = env.step(action)
            # Choose next action (epsilon-greedy)
            if random.random() < epsilon:
                next_action_idx = random.randrange(len(env.actions))
            else:
                next_action_idx = np.argmax(Q[next_state])
            # SARSA update
            td_target = reward + gamma * Q[next_state][next_action_idx]
            td_error = td_target - Q[state][action_idx]
            Q[state][action_idx] += alpha * td_error
            state = next_state
            action_idx = next_action_idx
            total_reward += reward
        episode_rewards.append(total_reward)
        epsilon = max(min_epsilon, epsilon * epsilon_decay)
    policy = {}
    for s in env.states:
        if s != env.terminal and s not in env.obstacles:
            policy[s] = env.actions[np.argmax(Q[s])]
    return Q, policy, episode_rewards